In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

# Number of beneficiaries and transactions to simulate
N_BENEFICIARIES = 2000
N_TRANSACTIONS = 20000

# Simulate beneficiaries with a fixed scheme amount each
schemes = {
    "Old Age Pension": 2000,
    "Scholarship": 5000,
    "Disability Pension": 3000,
    "Widow Pension": 2500
}

beneficiaries = pd.DataFrame({
    "beneficiary_id": [f"BEN{i:05d}" for i in range(N_BENEFICIARIES)],
    "scheme": np.random.choice(list(schemes.keys()), N_BENEFICIARIES),
    "district": np.random.choice(["District A", "District B", "District C", "District D", "District E"], N_BENEFICIARIES),
    "bank_account": [f"ACC{np.random.randint(10**9, 10**10)}" for _ in range(N_BENEFICIARIES)]
})

beneficiaries["scheme_amount"] = beneficiaries["scheme"].map(schemes)

beneficiaries.head()

ValueError: high is out of bounds for int32

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

# Number of beneficiaries and transactions to simulate
N_BENEFICIARIES = 2000
N_TRANSACTIONS = 20000

# Simulate beneficiaries with a fixed scheme amount each
schemes = {
    "Old Age Pension": 2000,
    "Scholarship": 5000,
    "Disability Pension": 3000,
    "Widow Pension": 2500
}

beneficiaries = pd.DataFrame({
    "beneficiary_id": [f"BEN{i:05d}" for i in range(N_BENEFICIARIES)],
    "scheme": np.random.choice(list(schemes.keys()), N_BENEFICIARIES),
    "district": np.random.choice(["District A", "District B", "District C", "District D", "District E"], N_BENEFICIARIES),
    # Fixed: Use dtype=np.int64 to handle larger integers, or reduce the range
    "bank_account": [f"ACC{np.random.randint(10**8, 10**9, dtype=np.int64)}" for _ in range(N_BENEFICIARIES)]
})

beneficiaries["scheme_amount"] = beneficiaries["scheme"].map(schemes)

beneficiaries.head()

,beneficiary_id,scheme,district,bank_account,scheme_amount
0,BEN00000,Disability Pension,District D,ACC406798051,3000
1,BEN00001,Widow Pension,District C,ACC635942599,2500
2,BEN00002,Old Age Pension,District E,ACC334751541,2000
3,BEN00003,Disability Pension,District A,ACC408274739,3000
4,BEN00004,Disability Pension,District E,ACC688800659,3000


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

# Number of beneficiaries and transactions to simulate
N_BENEFICIARIES = 2000
N_TRANSACTIONS = 20000

# Simulate beneficiaries with a fixed scheme amount each
schemes = {
    "Old Age Pension": 2000,
    "Scholarship": 5000,
    "Disability Pension": 3000,
    "Widow Pension": 2500
}

beneficiaries = pd.DataFrame({
    "beneficiary_id": [f"BEN{i:05d}" for i in range(N_BENEFICIARIES)],
    "scheme": np.random.choice(list(schemes.keys()), N_BENEFICIARIES),
    "district": np.random.choice(["District A", "District B", "District C", "District D", "District E"], N_BENEFICIARIES),
    "bank_account": [f"ACC{random.randint(10**9, 10**10 - 1)}" for _ in range(N_BENEFICIARIES)]
})

beneficiaries["scheme_amount"] = beneficiaries["scheme"].map(schemes)

beneficiaries.head()

,beneficiary_id,scheme,district,bank_account,scheme_amount
0,BEN00000,Disability Pension,District D,ACC3746317213,3000
1,BEN00001,Widow Pension,District C,ACC9697354961,2500
2,BEN00002,Old Age Pension,District E,ACC2181241943,2000
3,BEN00003,Disability Pension,District A,ACC1958682846,3000
4,BEN00004,Disability Pension,District E,ACC4163119785,3000


In [4]:
# Simulate monthly transactions over 12 months for each beneficiary
transactions = []
start_date = datetime(2025, 1, 1)

for _, row in beneficiaries.iterrows():
    for month in range(12):
        txn_date = start_date + timedelta(days=30 * month) + timedelta(days=random.randint(-2, 2))
        # Normal transactions: amount close to their scheme amount, small natural variation
        amount = row["scheme_amount"] + np.random.normal(0, 5)  # tiny variation, like rounding
        transactions.append({
            "beneficiary_id": row["beneficiary_id"],
            "scheme": row["scheme"],
            "district": row["district"],
            "bank_account": row["bank_account"],
            "transaction_date": txn_date,
            "amount": round(amount, 2)
        })

df = pd.DataFrame(transactions)
print(f"Total transactions generated: {len(df)}")
df.head(10)

Total transactions generated: 24000


,beneficiary_id,scheme,district,bank_account,transaction_date,amount
0,BEN00000,Disability Pension,District D,ACC3746317213,2025-01-03,3001.10
1,BEN00000,Disability Pension,District D,ACC3746317213,2025-01-29,3006.45
2,BEN00000,Disability Pension,District D,ACC3746317213,2025-03-01,3002.13
3,BEN00000,Disability Pension,District D,ACC3746317213,2025-04-01,3011.09
4,BEN00000,Disability Pension,District D,ACC3746317213,2025-05-02,2992.18
5,BEN00000,Disability Pension,District D,ACC3746317213,2025-06-01,2998.91
6,BEN00000,Disability Pension,District D,ACC3746317213,2025-06-30,2994.58
7,BEN00000,Disability Pension,District D,ACC3746317213,2025-08-01,2994.92
8,BEN00000,Disability Pension,District D,ACC3746317213,2025-08-29,2994.96
9,BEN00000,Disability Pension,District D,ACC3746317213,2025-09-30,3000.84


In [5]:
# Mark all transactions as legitimate by default
df["is_fraud"] = 0

fraud_records = []

# --- Pattern 1: Duplicate payouts (same beneficiary paid twice within days) ---
dup_sample = df.sample(150, random_state=1)
for _, row in dup_sample.iterrows():
    dup = row.copy()
    dup["transaction_date"] = row["transaction_date"] + timedelta(days=random.randint(1, 3))
    dup["is_fraud"] = 1
    fraud_records.append(dup)

# --- Pattern 2: Amount anomalies (inflated payout, 3-10x normal) ---
amount_sample = df.sample(150, random_state=2)
for idx in amount_sample.index:
    multiplier = random.uniform(3, 10)
    df.loc[idx, "amount"] = round(df.loc[idx, "amount"] * multiplier, 2)
    df.loc[idx, "is_fraud"] = 1

# --- Pattern 3: Frequency spikes (extra unexpected payouts) ---
freq_sample = beneficiaries.sample(100, random_state=3)
for _, row in freq_sample.iterrows():
    for _ in range(random.randint(3, 6)):
        extra_date = start_date + timedelta(days=random.randint(0, 360))
        fraud_records.append({
            "beneficiary_id": row["beneficiary_id"],
            "scheme": row["scheme"],
            "district": row["district"],
            "bank_account": row["bank_account"],
            "transaction_date": extra_date,
            "amount": row["scheme_amount"],
            "is_fraud": 1
        })

# --- Pattern 4: Bank account mismatch (payout to a DIFFERENT account than usual) ---
account_sample = df.sample(100, random_state=4)
for idx in account_sample.index:
    df.loc[idx, "bank_account"] = f"ACC{random.randint(10**9, 10**10 - 1)}"  # random new account
    df.loc[idx, "is_fraud"] = 1

# Combine everything
fraud_df = pd.DataFrame(fraud_records)
df = pd.concat([df, fraud_df], ignore_index=True)
df = df.sort_values("transaction_date").reset_index(drop=True)

print(f"Total transactions: {len(df)}")
print(df["is_fraud"].value_counts())
print(f"Fraud rate: {df['is_fraud'].mean()*100:.2f}%")

AttributeError: 'dict' object has no attribute 'dtype'

In [6]:
# Mark all transactions as legitimate by default
df["is_fraud"] = 0

fraud_records = []

# --- Pattern 1: Duplicate payouts (same beneficiary paid twice within days) ---
dup_sample = df.sample(150, random_state=1)
for _, row in dup_sample.iterrows():
    # Convert Series to dictionary to maintain consistency
    dup = row.to_dict()
    dup["transaction_date"] = row["transaction_date"] + timedelta(days=random.randint(1, 3))
    dup["is_fraud"] = 1
    fraud_records.append(dup)

# --- Pattern 2: Amount anomalies (inflated payout, 3-10x normal) ---
amount_sample = df.sample(150, random_state=2)
for idx in amount_sample.index:
    multiplier = random.uniform(3, 10)
    df.loc[idx, "amount"] = round(df.loc[idx, "amount"] * multiplier, 2)
    df.loc[idx, "is_fraud"] = 1

# --- Pattern 3: Frequency spikes (extra unexpected payouts) ---
freq_sample = beneficiaries.sample(100, random_state=3)
for _, row in freq_sample.iterrows():
    for _ in range(random.randint(3, 6)):
        extra_date = start_date + timedelta(days=random.randint(0, 360))
        fraud_records.append({
            "beneficiary_id": row["beneficiary_id"],
            "scheme": row["scheme"],
            "district": row["district"],
            "bank_account": row["bank_account"],
            "transaction_date": extra_date,
            "amount": row["scheme_amount"],
            "is_fraud": 1
        })

# --- Pattern 4: Bank account mismatch (payout to a DIFFERENT account than usual) ---
account_sample = df.sample(100, random_state=4)
for idx in account_sample.index:
    df.loc[idx, "bank_account"] = f"ACC{random.randint(10**9, 10**10 - 1)}"  # random new account
    df.loc[idx, "is_fraud"] = 1

# Combine everything - ensure all fraud_records are dictionaries before creating DataFrame
fraud_df = pd.DataFrame(fraud_records)
df = pd.concat([df, fraud_df], ignore_index=True)
df = df.sort_values("transaction_date").reset_index(drop=True)

print(f"Total transactions: {len(df)}")
print(df["is_fraud"].value_counts())
print(f"Fraud rate: {df['is_fraud'].mean()*100:.2f}%")

Total transactions: 24581
is_fraud
0    23750
1      831
Name: count, dtype: int64
Fraud rate: 3.38%


In [7]:
df.to_csv('dbt_transactions.csv', index=False)
print("Saved dbt_transactions.csv")
print(df.shape)

Saved dbt_transactions.csv
(24581, 7)


In [8]:
df = df.sort_values(["beneficiary_id", "transaction_date"]).reset_index(drop=True)

# Feature 1: Days since this beneficiary's previous transaction
df["days_since_last_txn"] = df.groupby("beneficiary_id")["transaction_date"].diff().dt.days
df["days_since_last_txn"] = df["days_since_last_txn"].fillna(30)  # first transaction, assume normal gap

# Feature 2: Amount z-score PER BENEFICIARY (how unusual is this amount for THIS person)
beneficiary_stats = df.groupby("beneficiary_id")["amount"].agg(["mean", "std"]).reset_index()
beneficiary_stats.columns = ["beneficiary_id", "amount_mean", "amount_std"]
beneficiary_stats["amount_std"] = beneficiary_stats["amount_std"].fillna(1)  # avoid divide-by-zero

df = df.merge(beneficiary_stats, on="beneficiary_id", how="left")
df["amount_zscore"] = (df["amount"] - df["amount_mean"]) / df["amount_std"]

# Feature 3: Total transaction count per beneficiary (frequency)
txn_counts = df.groupby("beneficiary_id").size().reset_index(name="total_txn_count")
df = df.merge(txn_counts, on="beneficiary_id", how="left")

# Feature 4: Bank account mismatch (does this transaction's account match beneficiary's registered account?)
df = df.merge(beneficiaries[["beneficiary_id", "bank_account"]].rename(columns={"bank_account": "registered_account"}), on="beneficiary_id", how="left")
df["account_mismatch"] = (df["bank_account"] != df["registered_account"]).astype(int)

df[["beneficiary_id", "amount", "days_since_last_txn", "amount_zscore", "total_txn_count", "account_mismatch", "is_fraud"]].head(10)

,beneficiary_id,amount,days_since_last_txn,amount_zscore,total_txn_count,account_mismatch,is_fraud
0,BEN00000,3001.10,30.0,0.197361,12,0,0
1,BEN00000,3006.45,26.0,1.189573,12,0,0
2,BEN00000,3002.13,31.0,0.388385,12,0,0
3,BEN00000,3011.09,31.0,2.050109,12,0,0
4,BEN00000,2992.18,31.0,-1.456945,12,0,0
5,BEN00000,2998.91,30.0,-0.208797,12,0,0
6,BEN00000,2994.58,29.0,-1.011840,12,0,0
7,BEN00000,2994.92,32.0,-0.948784,12,0,0
8,BEN00000,2994.96,28.0,-0.941366,12,0,0
9,BEN00000,3000.84,32.0,0.149141,12,0,0


In [1]:
df.groupby("is_fraud")[["days_since_last_txn", "amount_zscore", "total_txn_count", "account_mismatch"]].mean()

NameError: name 'df' is not defined

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

df = pd.read_csv('dbt_transactions.csv', parse_dates=['transaction_date'])
print(df.shape)
df.head()

(24581, 7)


,beneficiary_id,scheme,district,bank_account,transaction_date,amount,is_fraud
0,BEN01410,Widow Pension,District A,ACC5578929439,2024-12-30,2507.17,0
1,BEN01635,Disability Pension,District A,ACC4786674545,2024-12-30,3005.42,0
2,BEN00641,Scholarship,District B,ACC7757700186,2024-12-30,4985.96,0
3,BEN01634,Old Age Pension,District D,ACC8906163641,2024-12-30,2001.13,0
4,BEN00639,Widow Pension,District D,ACC8997701619,2024-12-30,2496.29,0


In [3]:
df = df.sort_values(["beneficiary_id", "transaction_date"]).reset_index(drop=True)

df["days_since_last_txn"] = df.groupby("beneficiary_id")["transaction_date"].diff().dt.days
df["days_since_last_txn"] = df["days_since_last_txn"].fillna(30)

beneficiary_stats = df.groupby("beneficiary_id")["amount"].agg(["mean", "std"]).reset_index()
beneficiary_stats.columns = ["beneficiary_id", "amount_mean", "amount_std"]
beneficiary_stats["amount_std"] = beneficiary_stats["amount_std"].fillna(1)

df = df.merge(beneficiary_stats, on="beneficiary_id", how="left")
df["amount_zscore"] = (df["amount"] - df["amount_mean"]) / df["amount_std"]

txn_counts = df.groupby("beneficiary_id").size().reset_index(name="total_txn_count")
df = df.merge(txn_counts, on="beneficiary_id", how="left")

# Note: since we didn't save the original `beneficiaries` table, we need to derive
# each beneficiary's registered account as their MOST COMMON account instead
registered_accounts = df.groupby("beneficiary_id")["bank_account"].agg(lambda x: x.mode()[0]).reset_index()
registered_accounts.columns = ["beneficiary_id", "registered_account"]
df = df.merge(registered_accounts, on="beneficiary_id", how="left")
df["account_mismatch"] = (df["bank_account"] != df["registered_account"]).astype(int)

print("Features rebuilt")
df.shape

Features rebuilt


(24581, 14)

In [4]:
df.groupby("is_fraud")[["days_since_last_txn", "amount_zscore", "total_txn_count", "account_mismatch"]].mean()

,days_since_last_txn,amount_zscore,total_txn_count,account_mismatch
is_fraud,,,,
0,29.765095,-0.021026,12.291074,0.000000
1,16.493381,0.600935,14.714801,0.120337


In [5]:
df["abs_amount_zscore"] = df["amount_zscore"].abs()
df.groupby("is_fraud")[["abs_amount_zscore"]].mean()

,abs_amount_zscore
is_fraud,
0,0.756186
1,0.885796


In [6]:
from scipy import stats

# Recompute using median and MAD (Median Absolute Deviation) — robust to outliers
beneficiary_robust_stats = df.groupby("beneficiary_id")["amount"].agg(
    amount_median="median",
    amount_mad=lambda x: (x - x.median()).abs().median()
).reset_index()

# Avoid divide-by-zero: if MAD is 0, use a small floor value
beneficiary_robust_stats["amount_mad"] = beneficiary_robust_stats["amount_mad"].replace(0, 1)

df = df.drop(columns=["amount_mean", "amount_std"], errors="ignore").merge(beneficiary_robust_stats, on="beneficiary_id", how="left")

# Robust z-score using median/MAD instead of mean/std
df["amount_zscore"] = (df["amount"] - df["amount_median"]) / (df["amount_mad"] * 1.4826)  # 1.4826 makes MAD comparable to std
df["abs_amount_zscore"] = df["amount_zscore"].abs()

df.groupby("is_fraud")[["abs_amount_zscore"]].mean()

,abs_amount_zscore
is_fraud,
0,0.886868
1,5092.610479


In [7]:
from sklearn.ensemble import IsolationForest

# Select the features that will feed the model
feature_cols = ["days_since_last_txn", "abs_amount_zscore", "total_txn_count", "account_mismatch"]
X = df[feature_cols]

# contamination = expected proportion of anomalies (we know it's ~3.38% from our injection)
iso_forest = IsolationForest(contamination=0.034, random_state=42, n_jobs=-1)
iso_forest.fit(X)

# Predict: -1 = anomaly (flagged as fraud), 1 = normal
df["anomaly_prediction"] = iso_forest.predict(X)
df["is_predicted_fraud"] = (df["anomaly_prediction"] == -1).astype(int)

print(df["is_predicted_fraud"].value_counts())

is_predicted_fraud
0    23745
1      836
Name: count, dtype: int64


In [8]:
from sklearn.metrics import classification_report, confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(df["is_fraud"], df["is_predicted_fraud"]))
print()
print("Classification Report:")
print(classification_report(df["is_fraud"], df["is_predicted_fraud"]))

Confusion Matrix:
[[23465   285]
 [  280   551]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     23750
           1       0.66      0.66      0.66       831

    accuracy                           0.98     24581
   macro avg       0.82      0.83      0.82     24581
weighted avg       0.98      0.98      0.98     24581



In [9]:
# Look at a few TRUE POSITIVES: fraud correctly caught
true_positives = df[(df["is_fraud"] == 1) & (df["is_predicted_fraud"] == 1)]
print("Sample of correctly caught fraud:")
true_positives[["beneficiary_id", "amount", "days_since_last_txn", "abs_amount_zscore", "total_txn_count", "account_mismatch"]].head(5)

Sample of correctly caught fraud:


,beneficiary_id,amount,days_since_last_txn,abs_amount_zscore,total_txn_count,account_mismatch
39,BEN00003,3000.00,11.0,0.000000,15,0
44,BEN00003,3000.00,5.0,0.000000,15,0
111,BEN00008,3003.93,2.0,0.674491,13,0
129,BEN00010,102696.22,31.0,40025.642693,13,0
135,BEN00010,3003.31,1.0,0.674491,13,0


In [10]:
false_positives = df[(df["is_fraud"] == 0) & (df["is_predicted_fraud"] == 1)]
print("Sample of false positives (normal transactions incorrectly flagged):")
false_positives[["beneficiary_id", "amount", "days_since_last_txn", "abs_amount_zscore", "total_txn_count", "account_mismatch"]].head(5)

Sample of false positives (normal transactions incorrectly flagged):


,beneficiary_id,amount,days_since_last_txn,abs_amount_zscore,total_txn_count,account_mismatch
49,BEN00003,2999.07,9.0,0.318414,15,0
710,BEN00058,2992.57,30.0,2.799702,18,0
711,BEN00058,2993.01,30.0,2.633905,18,0
712,BEN00058,3007.58,28.0,2.856223,18,0
714,BEN00058,2987.41,28.0,4.744044,18,0


In [11]:
from sklearn.neighbors import LocalOutlierFactor

# LOF doesn't have a separate .fit() then .predict() for novelty=False mode —
# fit_predict does both in one step, using the same contamination rate
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.034)
lof_predictions = lof.fit_predict(X)

df["lof_prediction"] = lof_predictions
df["is_predicted_fraud_lof"] = (df["lof_prediction"] == -1).astype(int)

print(df["is_predicted_fraud_lof"].value_counts())

is_predicted_fraud_lof
0    23745
1      836
Name: count, dtype: int64


C:\Users\bhoom\anaconda3\Lib\site-packages\sklearn\neighbors\_lof.py:327: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


In [12]:
print("LOF Confusion Matrix:")
print(confusion_matrix(df["is_fraud"], df["is_predicted_fraud_lof"]))
print()
print("LOF Classification Report:")
print(classification_report(df["is_fraud"], df["is_predicted_fraud_lof"]))

LOF Confusion Matrix:
[[23071   679]
 [  674   157]]

LOF Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97     23750
           1       0.19      0.19      0.19       831

    accuracy                           0.94     24581
   macro avg       0.58      0.58      0.58     24581
weighted avg       0.95      0.94      0.95     24581



In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lof_scaled = LocalOutlierFactor(n_neighbors=20, contamination=0.034)
lof_scaled_predictions = lof_scaled.fit_predict(X_scaled)

df["is_predicted_fraud_lof_scaled"] = (lof_scaled_predictions == -1).astype(int)

print("LOF (scaled features) Confusion Matrix:")
print(confusion_matrix(df["is_fraud"], df["is_predicted_fraud_lof_scaled"]))
print()
print("LOF (scaled features) Classification Report:")
print(classification_report(df["is_fraud"], df["is_predicted_fraud_lof_scaled"]))

LOF (scaled features) Confusion Matrix:
[[23056   694]
 [  689   142]]

LOF (scaled features) Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97     23750
           1       0.17      0.17      0.17       831

    accuracy                           0.94     24581
   macro avg       0.57      0.57      0.57     24581
weighted avg       0.94      0.94      0.94     24581



In [14]:
import joblib

# Save the trained Isolation Forest model (our chosen final model)
joblib.dump(iso_forest, 'isolation_forest_model.pkl')

# Save the feature columns list (so the dashboard/API knows what to expect later)
joblib.dump(feature_cols, 'feature_cols.pkl')

# Save the full dataframe with all engineered features and predictions
df.to_csv('dbt_transactions_with_predictions.csv', index=False)

print("Saved: isolation_forest_model.pkl, feature_cols.pkl, dbt_transactions_with_predictions.csv")

Saved: isolation_forest_model.pkl, feature_cols.pkl, dbt_transactions_with_predictions.csv
